> **About this notebook**
>
> This notebook is part of the example gallery of [CLABTOOLKIT](https://github.com/connectomicslab/clabtoolkit), an open-source Python toolkit for routine processing, manipulation and visualization of neuroimaging data. It shows how to use the `segmentationtools` module.
>
> 📦 **Install:** `pip install clabtoolkit` ([PyPI](https://pypi.org/project/clabtoolkit/)) · 📖 **Docs:** [clabtoolkit.readthedocs.io](https://clabtoolkit.readthedocs.io) · 📄 **Paper:** [arXiv:2607.02638](https://arxiv.org/abs/2607.02638)
>
> **Author:** Yasser Alemán-Gómez (Connectomics Lab, Radiology Department, CHUV – Lausanne University Hospital)

## Testing Methods in the `SEGMENTATIONTOOLS` Module

This Jupyter Notebook serves as a platform to rigorously test the various methods included in the Python module `SEGMENTATIONTOOLS`.

This module is particularly valuable for researchers segmenting and parcellating anatomical images:
propagating an atlas to an individual image through ANTs registration and transformation, and building
the lookup tables used to name and color the resulting tissue segmentations.

The tests are organized into the following sections for clarity and ease of navigation:

### Table of Contents

---

* 🔷 1. [`Image Segmentation`](#seg-segmentation). Methods dedicated to image segmentation and to the tables describing the resulting tissues
    * 🔸 1.1. [`abased_parcellation`](#seg-abased_parcellation). Perform an atlas-based parcellation using ANTs registration and transformation
    * 🔸 1.2. [`tissue_seg_table`](#seg-tissue_seg_table). Create a standard tissue segmentation lookup table

---

---
## <a id="seg-segmentation"></a>1. Image Segmentation
Methods dedicated to image segmentation and to the tables describing the resulting tissues.

### 1.1 `abased_parcellation`
<a id="seg-abased_parcellation"></a>

Perform an atlas-based parcellation using ANTs registration and transformation.

### Atlas-based parcellation with `abased_parcellation`

> ⚠️ **Requires ANTs.** This example is not executed here because it needs `antsRegistrationSyN.sh` and `antsApplyTransforms`. You can install ANTs locally, or run it inside a container with `cont_tech="singularity"` or `"docker"` and `cont_image` set.

`abased_parcellation` registers a subject's T1-weighted image to a template. It then uses the inverse transformation to bring a template-space atlas into the subject's native space.

**Workflow**

1. **Prepare output folders.** The function creates the folders for the transforms (`xfm_output`) and for the parcellation (`out_parc`) if they don't exist.
2. **Register the subject to the template.** It runs `antsRegistrationSyN.sh` with `-t s` (rigid + affine + SyN). The subject T1 is the moving image and the template is the fixed image.
3. **Rename the transforms (BIDS-style).** The ANTs outputs get the `desc-affine`, `desc-warp` and `desc-iwarp` entities and the `_xfm` suffix:
   - `<xfm_output>_desc-affine_xfm.mat`
   - `<xfm_output>_desc-warp_xfm.nii.gz`
   - `<xfm_output>_desc-iwarp_xfm.nii.gz`

   If the inverse warp already exists, registration is skipped unless `overwrite=True`.
4. **Bring the atlas into subject space.** `antsApplyTransforms` applies the inverse warp and the inverted affine (`[affine,1]`), using the subject T1 as reference:
   - **`atlas_type="spam"`**: the 4D probability maps are transformed volume by volume (`-e 3`) with the chosen `interp` (default `Linear`), so probabilities are preserved.
   - **`atlas_type="maxprob"`**: the discrete labels are transformed with `NearestNeighbor`. The result is then re-saved as a `Parcellation` with its color lookup table (`.lut` and `.tsv`).
5. **Clean up.** The intermediate `*Warped.nii.gz` images are deleted.

**Example**

```python
from clabtoolkit import segmentationtools as cltseg

out_parc = cltseg.abased_parcellation(
    t1="sub-01/anat/sub-01_T1w.nii.gz",
    t1_temp="templates/MNI152_T1_1mm.nii.gz",
    atlas="atlases/atlas-AAL_desc-maxprob_dseg.nii.gz",
    out_parc="derivatives/sub-01/anat/sub-01_space-T1w_atlas-AAL_dseg.nii.gz",
    xfm_output="derivatives/sub-01/xfm/sub-01_from-T1w_to-MNI152",
    atlas_type="maxprob",
    cont_tech="local",   # or "singularity" / "docker" with cont_image
)
```

**Outputs**
- Subject-space parcellation at `out_parc`. For a maxprob atlas, its `.lut` and `.tsv` lookup tables are written alongside it.
- Three reusable transforms in the `xfm_output` folder: affine, forward warp and inverse warp.

### 1.2 `tissue_seg_table`
<a id="seg-tissue_seg_table"></a>

Create a standard tissue segmentation lookup table.

In [ ]:
########################### Testing tissue_seg_table ###########################
import os
import tempfile
from pathlib import Path

import pandas as pd
import clabtoolkit.segmentationtools as cltseg

print("Test 1: Creating the tissue segmentation table without saving it...")
seg_df = cltseg.tissue_seg_table()
print(seg_df)
print("-------------------------------")
print(" ")

with tempfile.TemporaryDirectory() as tmp_dir:

    print("Test 2: Saving the table to a TSV file (str path)...")
    tsv_file = os.path.join(tmp_dir, "tissues.tsv")
    cltseg.tissue_seg_table(tsv_file)
    tsv_df = pd.read_csv(tsv_file, sep="\t")
    print(f"  File created: {os.path.isfile(tsv_file)}")
    print(f"  Saved table matches the returned table: {tsv_df.equals(seg_df)}")
    print("-------------------------------")
    print(" ")

    print("Test 3: Saving the table to a TSV file (Path object)...")
    tsv_path = Path(tmp_dir) / "tissues_path.tsv"
    cltseg.tissue_seg_table(tsv_path)
    print(f"  File created: {tsv_path.is_file()}")
    print("-------------------------------")
    print(" ")

    print("Test 4: Saving to a directory that does not exist...")
    bad_file = os.path.join(tmp_dir, "missing_dir", "tissues.tsv")
    try:
        cltseg.tissue_seg_table(bad_file)
    except FileNotFoundError as e:
        print(f"  Expected error raised: {e}")
    print("-------------------------------")